<a href="https://colab.research.google.com/github/peperjet/deep-learning/blob/main/RNN_Text_Classification/fast_text_260413.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

워드임베딩 - 패스트텍스트(FastText)

In [32]:
# Mecab 설치
!git clone http://github.com/SOMJANG/Mecab-ko-for-Google-Colab.git
%cd Mecab-ko-for-Google-Colab
!bash install_mecab-ko_on_colab190912.sh

Cloning into 'Mecab-ko-for-Google-Colab'...
remote: Enumerating objects: 138, done.
remote: Counting objects: 100% (47/47), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 138 (delta 26), reused 22 (delta 8), pack-reused 91 (from 1)
Receiving objects: 100% (138/138), 1.72 MiB | 109.00 KiB/s, done.
Resolving deltas: 100% (65/65), done.
/content/Mecab-ko-for-Google-Colab
Installing konlpy.....
Done
Installing mecab-0.996-ko-0.9.2.tar.gz.....
from https://bitbucket.org/eunjeon/mecab-ko/downloads/mecab-0.996-ko-0.9.2.tar.gz
--2026-04-13 14:09:11--  https://bitbucket.org/eunjeon/mecab-ko/downloads/mecab-0.996-ko-0.9.2.tar.gz
Resolving bitbucket.org (bitbucket.org)... 185.166.143.49, 185.166.143.48, 185.166.143.50, ...
Connecting to bitbucket.org (bitbucket.org)|185.166.143.49|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-04-13 14:09:11 ERROR 404: Not Found.

Done
Unpacking mecab-0.996-ko-0.9.2.tar.gz.......
Done
Change Directory to mecab-

In [33]:
pip install hgtk

자모 단위 한국어  FastText학습하기 : 네이버 쇼핑 리뷰 데이터로드

In [68]:
# fasttext 설치
!git clone https://github.com/facebookresearch/fastText.git
%cd fastText
!make
!pip install .

Cloning into 'fastText'...
remote: Enumerating objects: 3998, done.
remote: Total 3998 (delta 0), reused 0 (delta 0), pack-reused 3998 (from 1)
Receiving objects: 100% (3998/3998), 8.30 MiB | 224.00 KiB/s, done.
Resolving deltas: 100% (2527/2527), done.
/content/Mecab-ko-for-Google-Colab/fastText/fastText
c++ -pthread -std=c++17 -march=native -O3 -funroll-loops -DNDEBUG -c src/args.cc
c++ -pthread -std=c++17 -march=native -O3 -funroll-loops -DNDEBUG -c src/autotune.cc
c++ -pthread -std=c++17 -march=native -O3 -funroll-loops -DNDEBUG -c src/matrix.cc
c++ -pthread -std=c++17 -march=native -O3 -funroll-loops -DNDEBUG -c src/dictionary.cc
c++ -pthread -std=c++17 -march=native -O3 -funroll-loops -DNDEBUG -c src/loss.cc
c++ -pthread -std=c++17 -march=native -O3 -funroll-loops -DNDEBUG -c src/productquantizer.cc
c++ -pthread -std=c++17 -march=native -O3 -funroll-loops -DNDEBUG -c src/densematrix.cc
c++ -pthread -std=c++17 -march=native -O3 -funroll-loops -DNDEBUG -c src/quantmatrix.cc
c++ -pt

In [35]:
import re
import pandas as pd
import urllib.request
from tqdm import tqdm
import hgtk
from konlpy.tag import Mecab

In [36]:
urllib.request.urlretrieve("https://raw.githubusercontent.com/bab2min/corpus/master/sentiment/naver_shopping.txt", filename="/content/naver_shopping.txt")

('/content/naver_shopping.txt', <http.client.HTTPMessage at 0x78a3e8aebfb0>)

In [37]:
total_data = pd.read_table('/content/naver_shopping.txt', names=['ratings', 'reviews'])
print('전체 리뷰 개수 :',len(total_data)) #전체 리뷰 개수 출력

전체 리뷰 개수 : 200000


In [38]:
# 상위 5개 샘플 출력
total_data[:5]

,ratings,reviews
0,5,배공빠르고 굿
1,2,택배가 엉망이네용 저희집 밑에층에 말도없이 놔두고가고
2,5,아주좋아요 바지 정말 좋아서2개 더 구매했어요 이가격에 대박입니다. 바느질이 조금 ...
3,2,선물용으로 빨리 받아서 전달했어야 하는 상품이었는데 머그컵만 와서 당황했습니다. 전...
4,5,민트색상 예뻐요. 옆 손잡이는 거는 용도로도 사용되네요 ㅎㅎ


In [39]:
# 한글인지 체크
hgtk.checker.is_hangul('ㄱ')

True

In [40]:
# 한글인지 체크
hgtk.checker.is_hangul('28')

False

In [41]:
# 음절을 초성, 중성, 종성으로 분해
hgtk.letter.decompose('남')

('ㄴ', 'ㅏ', 'ㅁ')

In [42]:
# 초성, 중성을 경합
hgtk.letter.compose('ㄴ','ㅏ')

'나'

In [43]:
# 초성, 중성, 종성을 결합
hgtk.letter.compose('ㄴ','ㅏ','ㅁ')

'남'

In [46]:
# 한글이 아닌 입력에 대해서 에러 발생
hgtk.letter.decompose('1')

NotHangulException: 

In [47]:
# 결합할 수 없는 상황에서는 에러 발생
hgtk.letter.decompose('ㄴ','ㅏ','ㅁ')

TypeError: decompose() takes 1 positional argument but 3 were given

자모 단위 토큰화


In [48]:
def word_to_jamo(token):
  def to_special_token(jamo):
    if not jamo:
      return '-'
    else:
      return jamo

  decomposed_token = ''
  for char in token:
    try:
      #  char(음절)을 초성, 중성, 종성으로 분리
      cho, jung, jong = hgtk.letter.decompose(char)

      # 자모가 빈 문자일 경우 특수문자 - 로 대체
      cho = to_special_token(cho)
      jung = to_special_token(jung)
      jong = to_special_token(jong)
      decomposed_token = decomposed_token + cho + jung + jong


    # 만약 char(음절)이 한글이 아닐 경우 자모를 나누지 않고 추가
    except Exception as exception:
      if type(exception).__name__ == 'NotHangulException':
        decomposed_token = decomposed_token + char

  # 단어 토큰의 자모 단위 분리 결과를 추가
  return decomposed_token

In [49]:
word_to_jamo('남동생')

'ㄴㅏㅁㄷㅗㅇㅅㅐㅇ'

In [50]:
word_to_jamo('여동생')

'ㅇㅕ-ㄷㅗㅇㅅㅐㅇ'

In [51]:
# 1. 기존 제거
!pip uninstall -y konlpy mecab-python3
!rm -rf /usr/local/lib/mecab
!rm -rf /usr/local/include/mecab
!rm -rf /usr/local/bin/mecab
!rm -rf /var/lib/mecab/dic/mecab-ko-dic

# 2. 필수 도구
!apt-get update -qq
!apt-get install -y curl build-essential python3-dev git

# 3. mecab 설치
!bash <(curl -s https://raw.githubusercontent.com/konlpy/konlpy/master/scripts/mecab.sh)

# 4. konlpy만 설치
!pip install -q konlpy

# 5. 테스트
from konlpy.tag import Mecab
mecab = Mecab()
print(mecab.morphs('선물용으로 빨리 받아서 전달했어야 하는 상품이었는데 머그컵만 와서 당황했습니다.'))

Found existing installation: konlpy 0.6.0
Uninstalling konlpy-0.6.0:
  Successfully uninstalled konlpy-0.6.0
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
build-essential is already the newest version (12.9ubuntu3).
curl is already the newest version (7.81.0-1ubuntu1.23).
git is already the newest version (1:2.34.1-1ubuntu1.17).
python3-dev is already the newest version (3.10.6-1~22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 0 not upgraded.
Install mecab-ko
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 1381k  100 1381k    0     0   900k      0  0:0

In [52]:
# 환경 변수가 올바르게 설정되어 있으면 Mecab 객체가 자동으로 사전을 찾을 수 있습니다.
mecab = Mecab()
print(mecab.morphs('선물용으로 빨리 받아서 전달했어야 하는 상품이었는데 머그컵만 와서 당황했습니다.'))

['선물', '용', '으로', '빨리', '받', '아서', '전달', '했어야', '하', '는', '상품', '이', '었', '는데', '머그', '컵', '만', '와서', '당황', '했', '습니다', '.']


In [53]:
def word_to_jamo(token):
    def to_special_token(jamo):
        return '-' if not jamo else jamo

    decomposed_token = ''
    for char in token:
        try:
            cho, jung, jong = hgtk.letter.decompose(char)
            cho = to_special_token(cho)
            jung = to_special_token(jung)
            jong = to_special_token(jong)
            decomposed_token += cho + jung + jong
        except Exception as exception:
            if type(exception).__name__ == 'NotHangulException':
                decomposed_token += char
    return decomposed_token


In [54]:
def tokenize_by_jamo(text):
    return [word_to_jamo(token) for token in mecab.morphs(text)]


In [55]:
print(tokenize_by_jamo('선물용으로 빨리 받아서 전달했어야 하는 상품이었는데 머그컵만 와서 당황했습니다.'))

['ㅅㅓㄴㅁㅜㄹ', 'ㅇㅛㅇ', 'ㅇㅡ-ㄹㅗ-', 'ㅃㅏㄹㄹㅣ-', 'ㅂㅏㄷ', 'ㅇㅏ-ㅅㅓ-', 'ㅈㅓㄴㄷㅏㄹ', 'ㅎㅐㅆㅇㅓ-ㅇㅑ-', 'ㅎㅏ-', 'ㄴㅡㄴ', 'ㅅㅏㅇㅍㅜㅁ', 'ㅇㅣ-', 'ㅇㅓㅆ', 'ㄴㅡㄴㄷㅔ-', 'ㅁㅓ-ㄱㅡ-', 'ㅋㅓㅂ', 'ㅁㅏㄴ', 'ㅇㅘ-ㅅㅓ-', 'ㄷㅏㅇㅎㅘㅇ', 'ㅎㅐㅆ', 'ㅅㅡㅂㄴㅣ-ㄷㅏ-', '.']


In [56]:
print(mecab.morphs('테스트'))

['테스트']


In [57]:
from tqdm import tqdm

tokenized_data = []

for sample in tqdm(total_data['reviews'].to_list()):
  tokenized_sample = tokenize_by_jamo(sample)
  tokenized_data.append(tokenized_sample)

100%|██████████| 200000/200000 [00:53<00:00, 3770.40it/s]


In [58]:
print(tokenized_data[0])

['ㅂㅐ-ㄱㅗㅇ', 'ㅃㅏ-ㄹㅡ-', 'ㄱㅗ-', 'ㄱㅜㅅ']


자모 단위 토큰화

In [63]:
def jamo_to_word(jamo_sequence):
    tokenized_jamo = []
    index = 0

    # 1. 자모를 3개씩 자르기
    while index < len(jamo_sequence):
        # 문자가 한글(정상적인 자모)이 아닐 경우
        if not hgtk.checker.is_hangul(jamo_sequence[index]):
            tokenized_jamo.append(jamo_sequence[index])
            index += 1

        # 문자가 정상적인 자모라면 초성, 중성, 종성을 하나의 토큰으로 간주
        else:
            tokenized_jamo.append(jamo_sequence[index:index + 3])
            index += 3

    # 2. 자모를 다시 단어로 복원
    word = ''
    try:
        for jamo in tokenized_jamo:
            # 초성, 중성, 종성 묶음인 경우
            if len(jamo) == 3:
                if jamo[2] == '-':
                    # 종성이 없는 경우
                    word = word + hgtk.letter.compose(jamo[0], jamo[1])
                else:
                    # 종성이 있는 경우
                    word = word + hgtk.letter.compose(jamo[0], jamo[1], jamo[2])
            else:
                # 한글이 아닌 경우
                word = word + jamo

    # 복원 중 에러 발생 시 원래 입력 반환
    except Exception as exception:
        if type(exception).__name__ == 'NotHangulException':
            return jamo_sequence

    return word

In [65]:
jamo_to_word('ㄴㅏㅁㄷㅗㅇㅅㅐㅇ')

'남동생'

In [69]:
import fasttext

In [71]:
with open('tokenized_data.txt', 'w') as out:
    for line in tqdm(tokenized_data, unit='line'):
        out.write(' '.join(line) + '\n')

100%|██████████| 200000/200000 [00:00<00:00, 370384.79line/s]
